# 11 — Model 4: Decline Classifier (XGBoost Binary Classification)

**Tujuan:** Menjawab pertanyaan inti project ini — **"Apakah video ini akan/sedang mengalami penurunan views?"**  
**Target:** `is_declining` (views_trend_ratio < 0.90) — distribusi seimbang: 50.9% declining / 49.1% normal  
**Algoritma:** XGBoost Classifier + threshold tuning + SHAP + probability score  
**Output:**
- `backend/models/model4_decline_classifier.pkl`
- `backend/scalers/scaler_model4.pkl`
- `backend/models/model4_threshold.pkl` (optimal probability threshold)
- `data/processed/model_output_decline.csv`

**Catatan:** Model ini adalah yang paling langsung menjawab business problem 'diagnosa penurunan views'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, auc, average_precision_score,
    f1_score, precision_score, recall_score, roc_curve
)
import shap

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
print('Libraries loaded ✓')

In [ ]:
# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────
fm = pd.read_csv('../../data/processed/features_merged.csv')
abis = pd.read_csv('../../data/cleaned/abis_cleaning.csv')[['video_id', 'ts1_views']]

df = fm.merge(abis, on='video_id', how='inner').dropna(subset=['ts1_views']).reset_index(drop=True)
df['ts1_views'] = df['ts1_views'].fillna(0)

print(f'Shape: {df.shape}')
print(f'\nTarget distribution (is_declining):')
print(df['is_declining'].value_counts(normalize=True).round(3))
print(f'\nTotal declining videos: {df["is_declining"].sum()} / {len(df)}')

In [ ]:
# ── 2. EDA TARGET vs FITUR UTAMA ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('EDA — is_declining vs Key Features', fontsize=13, fontweight='bold')

key_features = [
    ('views_trend_ratio', 'Views Trend Ratio'),
    ('ts1_views',         'ts1 Views (log)'),
    ('ctr_vs_channel_avg','CTR vs Channel Avg'),
    ('engagement_score',  'Engagement Score'),
    ('like_rate',         'Like Rate'),
    ('rolling_mean_views_7d', 'Rolling Mean 7d'),
    ('video_age_days',    'Video Age Days'),
    ('revenue_idr_log',   'Revenue (log)'),
]

for ax, (feat, title) in zip(axes.flatten(), key_features):
    if feat == 'ts1_views':
        data0 = np.log1p(df.loc[df['is_declining']==0, feat])
        data1 = np.log1p(df.loc[df['is_declining']==1, feat])
    else:
        data0 = df.loc[df['is_declining']==0, feat]
        data1 = df.loc[df['is_declining']==1, feat]
    ax.hist(data0, bins=40, alpha=0.6, color='#2196F3', label='Normal (0)', density=True)
    ax.hist(data1, bins=40, alpha=0.6, color='#FF5722', label='Declining (1)', density=True)
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('../../data/processed/model4_eda.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3. FEATURE SELECTION (TANPA LEAKAGE) ─────────────────────────────────────
#
# Target `is_declining` = views_trend_ratio < 0.90
# views_trend_ratio = ema_views_5 / rolling_avg_views_15 (dari shift-1 data)
#
# DIHAPUS dari fitur:
# - views_trend_ratio (IS the derivation basis of the target — leakage)
# - views_vs_channel_avg, peak_views, is_viral, is_top_performer (views-derived)
# - watch_time_hours, views_volatility, subscriber_net (corr tinggi dengan views total)
# - views (total views = future data, tidak tersedia saat prediksi awal)
# - growth_1_to_2, growth_2_to_3, growth_3_to_4, growth_trend, avg_growth_rate
# - view_velocity (total views / age)
# - is_declining (the TARGET itself)
# - views_pct_change, views_deviation (correlated with trend_ratio)

LEAKAGE_FEATURES = [
    'views_trend_ratio',        # direct derivation of target
    'views_vs_channel_avg',     # total views / constant
    'peak_views',               # derived from ts columns
    'is_viral', 'is_top_performer',
    'watch_time_hours',         # views × duration
    'views_volatility',         # std(ts1-ts4)
    'subscriber_net',
    'views',                    # total views (future target)
    'growth_1_to_2', 'growth_2_to_3', 'growth_3_to_4',
    'growth_trend', 'avg_growth_rate',
    'view_velocity',
    'is_declining',             # this IS the target
    'views_pct_change',         # derived from ts3/ts4 (future data)
    'views_deviation',          # (views - rolling_avg) / rolling_std — uses total views
]

ALL_FEATS = [c for c in df.columns if c not in LEAKAGE_FEATURES + ['video_id']]
ALL_FEATS = [c for c in ALL_FEATS if df[c].dtype in [float, int, np.float64, np.int64]]

print(f'Kandidat fitur: {len(ALL_FEATS)}')
print(ALL_FEATS)

In [ ]:
# ── 4. ELIMINASI MULTIKOLINEARITAS (> 0.85) ───────────────────────────────────
X_raw = df[ALL_FEATS].copy().fillna(0).replace([np.inf, -np.inf], 0)
y = df['is_declining'].values

corr_matrix = X_raw.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = []
for col in upper.columns:
    hi_corr = upper.index[upper[col] > 0.85].tolist()
    for f in hi_corr:
        corr_col = abs(np.corrcoef(X_raw[col], y)[0, 1])
        corr_f   = abs(np.corrcoef(X_raw[f],   y)[0, 1])
        drop_cand = col if corr_col < corr_f else f
        if drop_cand not in to_drop:
            to_drop.append(drop_cand)

SELECTED = [f for f in ALL_FEATS if f not in to_drop]
print(f'Dibuang karena multikolinearitas ({len(to_drop)}): {to_drop}')
print(f'Fitur final ({len(SELECTED)}): {SELECTED}')

X = X_raw[SELECTED].copy()

In [ ]:
# ── 5. TRAIN/TEST SPLIT KRONOLOGIS 80/20 ─────────────────────────────────────
split_idx = int(len(df) * 0.80)
X_train_raw, X_test_raw = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train declining: {y_train.mean():.2%} | Test declining: {y_test.mean():.2%}')

In [ ]:
# ── 6. XGBOOST CLASSIFIER ─────────────────────────────────────────────────────
clf = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=5,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.1,
    reg_lambda=1.5,
    gamma=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    early_stopping_rounds=50,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=0,
)

clf.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

y_prob  = clf.predict_proba(X_test)[:, 1]
y_pred  = clf.predict(X_test)

print(f'Best iteration: {clf.best_iteration}')
print(f'\nDefault threshold (0.5):')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Declining'], digits=4))

In [ ]:
# ── 7. THRESHOLD TUNING (Optimal F1) ─────────────────────────────────────────
precision_arr, recall_arr, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-9)

optimal_idx       = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
y_pred_opt = (y_prob >= optimal_threshold).astype(int)

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc  = auc(recall_arr, precision_arr)

print(f'Optimal threshold: {optimal_threshold:.4f}')
print(f'ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}')
print(f'\nOptimal threshold report:')
print(classification_report(y_test, y_pred_opt, target_names=['Normal', 'Declining'], digits=4))

In [ ]:
# ── 8. STRATIFIED 5-FOLD CV ──────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
X_all_sc = scaler.transform(X)

cv_clf = XGBClassifier(
    n_estimators=clf.best_iteration + 1,
    learning_rate=0.05, max_depth=5, min_child_weight=5,
    subsample=0.80, colsample_bytree=0.80,
    reg_alpha=0.1, reg_lambda=1.5, gamma=0.1,
    use_label_encoder=False, eval_metric='logloss',
    random_state=RANDOM_SEED, n_jobs=-1, verbosity=0,
)

cv_roc_auc = cross_val_score(cv_clf, X_all_sc, y, cv=skf, scoring='roc_auc')
cv_f1      = cross_val_score(cv_clf, X_all_sc, y, cv=skf, scoring='f1')

print(f'5-Fold Stratified CV:')
print(f'  ROC-AUC: {cv_roc_auc.mean():.4f} ± {cv_roc_auc.std():.4f}')
print(f'  F1:      {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

In [ ]:
# ── 9. VISUALISASI KOMPREHENSIF ───────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)
fig.suptitle('Model 4 — Decline Classifier Evaluation', fontsize=14, fontweight='bold')

# (A) ROC Curve
ax1 = fig.add_subplot(gs[0, 0])
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax1.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC={roc_auc:.4f})')
ax1.plot([0,1],[0,1],'--',color='gray')
ax1.set_title('ROC Curve'); ax1.legend(); ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')

# (B) Precision-Recall Curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(recall_arr, precision_arr, color='#FF5722', lw=2, label=f'PR (AUC={pr_auc:.4f})')
ax2.axhline(y_test.mean(), color='gray', linestyle='--', label='Baseline')
ax2.axvline(recall_arr[optimal_idx], color='green', linestyle=':', label=f'Opt threshold={optimal_threshold:.3f}')
ax2.set_title('Precision-Recall Curve'); ax2.legend(fontsize=8)
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')

# (C) Confusion Matrix (Optimal Threshold)
ax3 = fig.add_subplot(gs[0, 2])
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=['Normal','Declining'], yticklabels=['Normal','Declining'])
ax3.set_title(f'Confusion Matrix (thr={optimal_threshold:.3f})')
ax3.set_xlabel('Predicted'); ax3.set_ylabel('Actual')

# (D) Probability Distribution
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(y_prob[y_test==0], bins=40, alpha=0.6, color='#2196F3', label='Normal', density=True)
ax4.hist(y_prob[y_test==1], bins=40, alpha=0.6, color='#FF5722', label='Declining', density=True)
ax4.axvline(optimal_threshold, color='green', linestyle='--', lw=2, label=f'Optimal thr')
ax4.set_title('Predicted Probability Distribution')
ax4.set_xlabel('P(declining)'); ax4.legend()

# (E) F1 vs Threshold
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(thresholds, f1_scores[:-1], color='purple', lw=2)
ax5.axvline(optimal_threshold, color='green', linestyle='--', label=f'Opt={optimal_threshold:.3f}')
ax5.set_title('F1 Score vs Threshold')
ax5.set_xlabel('Threshold'); ax5.set_ylabel('F1 Score'); ax5.legend()

# (F) CV Performance Summary
ax6 = fig.add_subplot(gs[1, 2])
metrics_names = ['ROC-AUC\n(Test)', 'PR-AUC\n(Test)', 'F1\n(Test, opt thr)', 'CV ROC-AUC\n(Mean)']
metrics_vals  = [
    roc_auc, pr_auc,
    f1_score(y_test, y_pred_opt),
    cv_roc_auc.mean()
]
colors = ['steelblue', '#FF5722', 'green', 'purple']
bars = ax6.bar(metrics_names, metrics_vals, color=colors, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, metrics_vals):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.4f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax6.set_ylim(0, 1.1)
ax6.set_title('Performance Summary')

plt.tight_layout()
plt.savefig('../../data/processed/model4_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10. SHAP ANALYSIS ─────────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# Untuk classifier biner: shap_values bisa berupa list atau array 2D
shap_for_plot = shap_values[1] if isinstance(shap_values, list) else shap_values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('SHAP Analysis — Model 4 Decline Classifier', fontsize=13, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(shap_for_plot, X_test_raw, feature_names=SELECTED,
                  plot_type='bar', max_display=15, show=False)
axes[0].set_title('Feature Importance (Mean |SHAP|)')

plt.sca(axes[1])
shap.summary_plot(shap_for_plot, X_test_raw, feature_names=SELECTED,
                  max_display=15, show=False)
axes[1].set_title('SHAP Value Distribution')

plt.tight_layout()
plt.savefig('../../data/processed/model4_shap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11. BUSINESS INSIGHT: RISK LEVEL SEGMENTATION ────────────────────────────
# Bagi probabilitas ke 3 level risiko untuk interpretasi yang lebih actionable
df_result = df[['video_id']].copy()

X_all_scaled = scaler.transform(X)
all_probs    = clf.predict_proba(X_all_scaled)[:, 1]

df_result['decline_prob']  = all_probs
df_result['decline_pred']  = (all_probs >= optimal_threshold).astype(int)
df_result['risk_level']    = pd.cut(
    all_probs,
    bins=[0, 0.30, 0.55, 0.75, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical']
)
df_result['is_declining_actual'] = df['is_declining'].values

print('Risk Level Distribution:')
print(df_result['risk_level'].value_counts())
print()
print('Accuracy per risk level (actual declining rate):')
print(df_result.groupby('risk_level')['is_declining_actual'].mean().round(3))

In [ ]:
# ── 12. SIMPAN ARTIFACTS ──────────────────────────────────────────────────────
os.makedirs('../../backend/models', exist_ok=True)
os.makedirs('../../backend/scalers', exist_ok=True)

joblib.dump(clf,                '../../backend/models/model4_decline_classifier.pkl')
joblib.dump(scaler,             '../../backend/scalers/scaler_model4.pkl')
joblib.dump(SELECTED,           '../../backend/models/model4_selected_features.pkl')
joblib.dump(float(optimal_threshold), '../../backend/models/model4_threshold.pkl')

df_result.to_csv('../../data/processed/model_output_decline.csv', index=False)

print('Artifacts saved ✓')
print(f'  model4_decline_classifier.pkl')
print(f'  scalers/scaler_model4.pkl')
print(f'  model4_selected_features.pkl ({len(SELECTED)} features)')
print(f'  model4_threshold.pkl (optimal={optimal_threshold:.4f})')
print(f'  model_output_decline.csv ({len(df_result)} rows)')

In [ ]:
# ── 13. RINGKASAN FINAL ───────────────────────────────────────────────────────
print('=' * 60)
print('MODEL 4 — DECLINE CLASSIFIER — RINGKASAN FINAL')
print('=' * 60)
print(f'  Fitur terpilih    : {len(SELECTED)}')
print(f'  Optimal threshold : {optimal_threshold:.4f}')
print(f'  ROC-AUC (test)    : {roc_auc:.4f}')
print(f'  PR-AUC (test)     : {pr_auc:.4f}')
print(f'  F1 (opt thr)      : {f1_score(y_test, y_pred_opt):.4f}')
print(f'  Precision (opt)   : {precision_score(y_test, y_pred_opt):.4f}')
print(f'  Recall (opt)      : {recall_score(y_test, y_pred_opt):.4f}')
print(f'  CV ROC-AUC        : {cv_roc_auc.mean():.4f} ± {cv_roc_auc.std():.4f}')
print(f'  CV F1             : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print()
print('Model ini menjawab core business question:')
print('"Apakah video ini mengalami/akan mengalami penurunan views?"')
print('Output: probability score 0-1 + risk level (Low/Medium/High/Critical)')